### Performance plots

In [ ]:
import numpy as np
from statplotannot.plots import fix_legend
from mab_colors import Palette2Arm
import seaborn as sns
from statplotannot.plots import Fig
import mab_subjects

perf_df = mab_subjects.GroupData().perf_all_corr_uncorr.latest

fig = Fig(3, 4, size=(8.5, 6), fontsize=11)

for i, y in enumerate(["All", "Easy", "Hard"]):
    ax = fig.subplot(fig.gs[i])
    sns.lineplot(
        data=perf_df,
        x="trial_id",
        y=y,
        hue="grp",
        palette=Palette2Arm().as_dict(),
        errorbar="se",
        estimator=np.mean,
    )
    fix_legend(ax)
    ax.set_ylim(0.4, 0.95)
    ax.set_ylabel("P(High)")
    ax.set_title(f"{y} combinations")
    ax.grid(axis="y", zorder=-1, alpha=0.5)

fig.savefig(mab_subjects.iapath / "choice_performance_easy_hard.pdf")

### Probability matrix

In [ ]:
import matplotlib.pyplot as plt
from neuropy.plotting import Fig
import seaborn as sns
from scipy.stats import binned_statistic_2d
from mab_colors import Palette2Arm
from statplotannot.plots import fix_legend
import mab_subjects
import numpy as np
from scipy.ndimage import gaussian_filter
from palettable.scientific.sequential import GrayC_7
from matplotlib.colors import BoundaryNorm, CenteredNorm
import matplotlib.ticker as ticker


bounds = np.linspace(0.4, 1, 15)
norm = BoundaryNorm(bounds, ncolors=256)

fig = Fig(6, 3, size=(8.5, 11), fontsize=11)

df = mab_subjects.GroupData().perf_probability_matrix.latest
# df = df[df["lesion"] == "pre_lesion"]
matrix = []
for g, grp in enumerate(["struc", "unstruc"]):
    df_grp = df[df["grp"] == grp]
    perf_mean = df_grp["perf_mat"].mean()
    # perf_mean = gaussian_filter(perf_mean, sigma=0.1)
    # perf_mean = np.tril(perf_mean, k=0)
    mask = np.triu(np.ones_like(perf_mean, dtype=bool), k=0)

    perf_mean[mask] = np.nan
    matrix.append(perf_mean)

    ax = fig.subplot(fig.gs[g])

    im = ax.pcolormesh(
        perf_mean.T,
        # cmap=GrayC_7.mpl_colormap,
        cmap="turbo",
        shading="auto",
        # vmin=0.6,
        # vmax=1,
        norm=norm,
    )
    ticks = np.arange(0, 6) + 0.5
    ax.set_xticks(ticks, [20, 30, 40, 60, 70, 80])
    ax.set_yticks(ticks, [20, 30, 40, 60, 70, 80])
    ax.set_xlim(1, 6)
    ax.set_ylim(0, 5)
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)
    ax.set_xlabel("Higher reward probability")
    ax.set_ylabel("Lower reward probability")

    cb = plt.colorbar(im, ax=ax, shrink=0.7)
    cb.set_label("P(High)")
    formatter = ticker.FormatStrFormatter("%.2f")
    cb.ax.yaxis.set_major_formatter(formatter)
    ax.set_title(f"{grp} sessions")

matrix_diff = matrix[0] - matrix[1]

ax = fig.subplot(fig.gs[2])
im = ax.pcolormesh(
    matrix_diff.T,
    # cmap=GrayC_7.mpl_colormap,
    cmap="bwr",
    shading="auto",
    norm=CenteredNorm(vcenter=0, halfrange=0.1),
)
ticks = np.arange(0, 6) + 0.5
ax.set_xticks(ticks, [20, 30, 40, 60, 70, 80])
ax.set_yticks(ticks, [20, 30, 40, 60, 70, 80])
ax.set_xlim(1, 6)
ax.set_ylim(0, 5)
ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(True)
ax.set_xlabel("Higher reward probability")
ax.set_ylabel("Lower reward probability")

cb = plt.colorbar(im, ax=ax, shrink=0.7)
cb.set_label("Diff.")
formatter = ticker.FormatStrFormatter("%.2f")
cb.ax.yaxis.set_major_formatter(formatter)
# ax.set_title(f"{grp} sessions\n(mean of last 99 trials)")

# fig.savefig(mab_subjects.iapath / "choice_performance_probability_matrix.pdf")

### Switching probability

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from neuropy import plotting
import mab_subjects
from mab_colors import Palette2Arm
from statplotannot.plots import Fig, SeabornPlotter
from statplotannot.plots.plot_utils import xtick_format


df = mab_subjects.GroupData().swp_by_quartiles.latest
palette = Palette2Arm(lightness_scale=1).as_dict()

fig = Fig(5, 5, fontsize=11)

ax = fig.subplot(fig.gs[0])

plot_kw = dict(
    data=df, x="Trials", y="swp", hue="grp", hue_order=["unstruc", "struc"], ax=ax
)
plotter = (
    SeabornPlotter(**plot_kw)
    .boxplot_filled(palette=palette)
    .stat_test(test_name="Kruskal")
)

ax.set_xlabel("Trial range")
ax.set_title("Switch probability")
ax.set_ylabel("P(Switch)")
ax.legend_.remove()
xtick_format(ax, rotation=30)

fig.savefig(mab_subjects.iapath / "mab_switching_probability_by_quartiles.pdf")